# Data, tokens, and masks

Prepare text without learning from padding.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

Batches contain sequences of different lengths. Padding fills missing positions, attention masks exclude them from context, and labels=-100 excludes them from cross entropy. These are different mechanisms.

SFT requires role boundaries and a clear choice of supervised tokens. Images, actions, and episodes also need consistent shapes, units, and time boundaries. Keep an episode entirely within one data split.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
samples = ['model', 'learn']
tokens = [list(s.encode('utf-8')) for s in samples]
width = max(map(len, tokens)) + 3
ids = torch.zeros(len(tokens), width, dtype=torch.long)
mask = torch.zeros_like(ids)
for row, sequence in enumerate(tokens):
    ids[row, :len(sequence)] = torch.tensor(sequence)
    mask[row, :len(sequence)] = 1
labels = ids.clone(); labels[mask == 0] = -100
logits = torch.randn(*ids.shape, 256)
loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=-100)
print('input_ids:', ids.tolist()); print('attention_mask:', mask.tolist())
print('labels:', labels.tolist()); print('masked loss:', loss.item())
assert (labels == -100).sum() == (mask == 0).sum()

## Try it yourself

Add non-ASCII text and compare UTF-8 bytes with character counts. This byte tokenizer is educational; real models require their own tokenizer and chat template.

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [PyTorch: CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)